In [2]:
%pip install duckdb

   ---------------------------------------- 0.0/13.2 MB ? eta -:--:--
   ---- ----------------------------------- 1.6/13.2 MB 11.8 MB/s eta 0:00:01
   ----------- ---------------------------- 3.9/13.2 MB 11.7 MB/s eta 0:00:01
   ------------------- -------------------- 6.3/13.2 MB 11.8 MB/s eta 0:00:01
   ------------------------- -------------- 8.4/13.2 MB 11.7 MB/s eta 0:00:01
   ------------------------------- -------- 10.5/13.2 MB 11.2 MB/s eta 0:00:01
   -------------------------------------- - 12.6/13.2 MB 11.2 MB/s eta 0:00:01
   ---------------------------------------  13.1/13.2 MB 11.1 MB/s eta 0:00:01
   ---------------------------------------  13.1/13.2 MB 11.1 MB/s eta 0:00:01
   ---------------------------------------  13.1/13.2 MB 11.1 MB/s eta 0:00:01
   ---------------------------------------  13.1/13.2 MB 11.1 MB/s eta 0:00:01
   ---------------------------------------  13.1/13.2 MB 11.1 MB/s eta 0:00:01
   ---------------------------------------- 13.2/13.2 MB 5.8 MB/s

In [3]:
import pandas as pd
import duckdb

# Load the processed data (not the raw data!)
df = pd.read_csv('../data/processed_inventory_data.csv')

# Test the connection
duckdb.query("SELECT * FROM df LIMIT 3").df()

,date,sku_id,warehouse_id,supplier_id,region,units_sold,inventory_level,supplier_lead_time_days,reorder_point,order_quantity,unit_cost,unit_price,promotion_flag,stockout_flag,demand_forecast,product_category,storage_cost_per_sqft,unit_square_footage,months_stagnant,total_holding_cost
0,2024-01-01,SKU_1,WH_1,SUP_8,West,10,592,14,379,0,13.95,20.48,0,0,8.52,apparel,0.15,1.0,1,88.8
1,2024-01-02,SKU_1,WH_1,SUP_8,West,17,575,14,379,0,13.95,20.48,0,0,18.63,apparel,0.15,1.0,2,172.5
2,2024-01-03,SKU_1,WH_1,SUP_8,North,35,540,14,379,0,13.95,20.48,1,0,39.62,apparel,0.15,1.0,8,648.0


### Phase 1: Data Integrity & Sanity Checks

In [5]:
query_sanity = """
SELECT 
    COUNT(*) AS total_rows,
    SUM(CASE WHEN inventory_level < 0 THEN 1 ELSE 0 END) AS negative_inventory_count,
    SUM(CASE WHEN total_holding_cost IS NULL THEN 1 ELSE 0 END) AS missing_cost_count
FROM df
"""
duckdb.query(query_sanity).df()

,total_rows,negative_inventory_count,missing_cost_count
0,91250,0.0,0.0


#### Insight:
Before executing financial aggregations, we must validate the logical and structural integrity of the dataset. In a production environment, corrupted data (such as negative physical inventory or uncalculated costs) will break downstream financial models and destroy stakeholder trust.

* **Audit Criteria:** 
  1. Checked for negative inventory counts (`inventory_level < 0`), which are physically impossible in a supply chain warehouse.
  2. Checked for missing or null holding costs (`total_holding_cost IS NULL`), which would distort our capital exposure calculations.
* **Validation Result:** Both checks returned `0` anomalies. The dataset is structurally sound, clean of logical impossibilities, and fully cleared for macro-level financial analysis.

### Phase 2 

In [18]:
query_baselines = """
SELECT 
    product_category,
    COUNT(DISTINCT sku_id) AS total_unique_skus,
    SUM(inventory_level) AS total_units_on_hand,
    ROUND(SUM(total_holding_cost), 0) AS "total_capital_tied_up_($)"
FROM df
GROUP BY product_category
ORDER BY "total_capital_tied_up_($)" DESC
"""
duckdb.query(query_baselines).df()

,product_category,total_unique_skus,total_units_on_hand,total_capital_tied_up_($)
0,furniture,11,"9,302,332","436,088,619"
1,heavy machinery,16,"13,601,685","81,711,201"
2,electronics,10,"8,689,605","14,787,858"
3,apparel,13,"11,432,789","9,380,443"


#### Insight:
The baseline analysis reveals a massive disparity in working capital allocation. 
* **The Bottleneck:** The `furniture` category represents the highest financial risk, with **$436M** in tied-up capital. 
* **The Driver:** Despite having fewer unique SKUs than heavy machinery or apparel, the physical footprint (unit square footage) of furniture drastically inflates its storage and holding costs over time. 
* **Strategic Next Step:** We must drill down into the furniture category to isolate the specific "dead" SKUs (stagnant for 6+ months) that are driving this cost, so procurement can target them for liquidation.

### Phase 3: Identifying the Dead Stock Outliers

In [20]:
query_dead_stock = """
SELECT 
    sku_id,
    warehouse_id,
    inventory_level,
    months_stagnant,
    ROUND(total_holding_cost, 0) AS "dead_capital_$"
FROM df
WHERE product_category = 'furniture' 
  AND months_stagnant >= 6
ORDER BY "dead_capital_$" DESC
LIMIT 10
"""
duckdb.query(query_dead_stock).df()

,sku_id,warehouse_id,inventory_level,months_stagnant,dead_capital_$
0,SKU_27,WH_4,966,11,"114,187"
1,SKU_35,WH_2,805,11,"112,163"
2,SKU_27,WH_4,816,11,"109,338"
3,SKU_28,WH_1,888,11,"108,456"
4,SKU_23,WH_4,803,11,"107,489"
5,SKU_21,WH_3,981,9,"106,506"
6,SKU_27,WH_2,867,10,"105,613"
7,SKU_50,WH_3,887,10,"105,545"
8,SKU_10,WH_4,761,11,"104,533"
9,SKU_23,WH_3,745,11,"101,462"


#### Insight:
Drilling into the `furniture` category for items stagnant > 6 months reveals high-cost outliers. However, the raw output shows duplicate SKU entries (e.g., `SKU_27` appearing multiple times). This indicates the dataset contains multiple transactional batches or shipments per SKU. To provide an actionable liquidation list, we must aggregate this data by SKU and Warehouse.

### Phase 4 : The Final Liquidation Target List

In [25]:
query_final_targets = """
SELECT 
    sku_id,
    warehouse_id,
    SUM(inventory_level) AS total_dead_units,
    ROUND(SUM(total_holding_cost), 0) AS "total_dead_capital_$"
FROM df
WHERE product_category = 'furniture' 
  AND months_stagnant >= 6
GROUP BY sku_id, warehouse_id
ORDER BY "total_dead_capital_$" DESC
LIMIT 10
"""
duckdb.query(query_final_targets).df()

,sku_id,warehouse_id,total_dead_units,total_dead_capital_$
0,SKU_49,WH_2,"113,053","8,227,692"
1,SKU_21,WH_4,"110,947","8,110,606"
2,SKU_27,WH_1,"110,876","7,794,018"
3,SKU_21,WH_2,"105,398","7,532,334"
4,SKU_35,WH_2,"100,512","7,509,432"
5,SKU_10,WH_1,"98,422","7,450,057"
6,SKU_43,WH_3,"100,609","7,336,929"
7,SKU_49,WH_4,"100,320","7,264,883"
8,SKU_43,WH_2,"96,730","7,123,952"
9,SKU_28,WH_5,"95,813","7,044,344"


#### Insight:
By aggregating the transactional batches by SKU and Warehouse, the true scale of the dead stock exposure becomes clear. 
* **The Prime Target:** `SKU_49` in `WH_2` is the single largest financial liability, tying up over **$8.2 Million** in working capital across 113,000+ stagnant units.
* **Business Action:** This aggregated list serves as the direct action plan for the Procurement and Warehouse teams. Liquidating just the top 3 items on this list at a 40% markdown would immediately free up over $14M in liquid cash.